# 🧠→📊 ISOM 260: Three Things a Chatbot Can't Do

**Session 3 — Bonus Workshop: Structure · Scale · Sight** | Suffolk University | Prof. Hasan Arslan

---

A chatbot gives you *prose*. Your code needs *data*. That single difference is why companies pay for the API and not for 500 ChatGPT seats.

In the next 15 minutes you will do three things no chat window can:

| # | Wow | What it means for a business |
|---|-----|------------------------------|
| 1 | **Structure** — the model returns guaranteed JSON that matches *your* schema | Output can flow straight into a spreadsheet, a database, a dashboard |
| 2 | **Scale** — 30 messy customer reviews in, 30 clean rows + a chart out, **one call** | The week-long analyst task becomes a cell you re-run every Monday |
| 3 | **Sight** — photograph a receipt with your phone, get an expense line item back | Paper → data. Invoices, whiteboards, business cards, menus |

### Before you run anything
Same setup as Wednesday: your key lives in the **🔑 Secrets** panel as `GOOGLE_API_KEY`. `File → Save a copy in Drive`, then run top to bottom. Total: **3 API calls**, free tier, $0.

In [ ]:
# ── Setup (same as Wednesday) ─────────────────────────────────────
!pip install -q google-genai

In [ ]:
# ── Key + client + imports ────────────────────────────────────────
from google.colab import userdata
from google import genai
from google.genai import types
from pydantic import BaseModel, Field
from typing import Literal
import pandas as pd, json

client = genai.Client(api_key=userdata.get("GOOGLE_API_KEY"))
MODEL = "gemini-2.5-flash"

def structured(contents, schema, temperature=0.2):
    '''One call that MUST come back as JSON matching `schema`.
    Extraction doesn't need the model to 'think out loud', so we turn
    thinking off (thinking_budget=0): faster, and roughly half the bill.'''
    r = client.models.generate_content(
        model=MODEL, contents=contents,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=schema,
            temperature=temperature,
            thinking_config=types.ThinkingConfig(thinking_budget=0),
        ))
    # r.parsed is already a Python object built from your schema
    return r.parsed, r.usage_metadata

print("✅ Ready. Same key, same model — new superpowers.")

## 📐 Part 1 — Structure: the schema is a contract

Wednesday your code got back a *string*. Useful to read, useless to a program. Watch what happens when you hand the model a **schema** — a Python class that says exactly what fields you want, what type each is, and which values are allowed.

The model is no longer *writing*. It is **filling in a form you designed.**

In [ ]:
# ── A messy customer email → a clean support ticket ──────────────
INBOUND_EMAIL = '''
Subject: this is the THIRD time

Hi, Priya Raman here (order BP-88213). My BrewPass bag arrived crushed AGAIN and the
app still says "delivered Tuesday" when it showed up Friday. I run a café pop-up on
weekends so the Tuesday date actually matters. I've been a subscriber since January
and I like the coffee, but if this happens once more I'm cancelling. Can someone
actually fix the delivery date issue and replace this bag?
'''

class Ticket(BaseModel):
    customer_name: str
    order_id: str | None = Field(description="Order or account reference if present")
    issue_category: Literal["delivery", "product_quality", "billing", "app_bug", "other"]
    urgency: int = Field(ge=1, le=5, description="5 = at risk of cancelling, 1 = FYI")
    sentiment: Literal["angry", "frustrated", "neutral", "positive"]
    requested_actions: list[str]
    one_line_summary: str = Field(description="For the support dashboard, under 15 words")

ticket, usage = structured(
    f"Read this customer email and fill in the support ticket exactly.\n\n{INBOUND_EMAIL}",
    schema=Ticket)

print(json.dumps(ticket.model_dump(), indent=2))
print("\n— this is a Python object now, not a paragraph —")
print(f"ticket.urgency = {ticket.urgency}  →  ", "🚨 route to a human" if ticket.urgency >= 4 else "auto-reply is fine")
print(f"tokens: {usage.prompt_token_count} in / {usage.candidates_token_count} out")

**Look at what just happened.** `issue_category` could only be one of five values you allowed. `urgency` is an integer between 1 and 5, so `ticket.urgency >= 4` is a line of code you can *route on*. No regex, no "please answer in JSON" begging, no parsing prose.

> 💼 **Boardroom line:** *"We don't ask the model for an answer. We hand it a form and it fills it in."* That sentence is the difference between an AI demo and an AI system.

---

## 📊 Part 2 — Scale: 30 reviews, one call, one chart

Here are 30 real-sounding reviews for **BrewPass**, a fictional campus coffee subscription. Reading them takes a person 10 minutes; tagging them consistently takes an hour and they'll still disagree with themselves. Your code will do it in one call and draw the chart the product team actually needs.

In [ ]:
# ── 30 messy reviews (the raw material) ──────────────────────────
REVIEWS = [
 "Love the beans, hate the app. Crashed twice while I was trying to pause my plan.",
 "$24/month is steep for a student. Coffee's good but I might cancel before finals.",
 "Bag came a week late and half open. Second time. Not great.",
 "Honestly the best cold brew I've had near campus. Zero complaints.",
 "Customer service replied in 4 minutes and refunded the crushed bag. Impressed.",
 "Why can't I change my roast preference without emailing someone?? It's 2026.",
 "Tastes burnt. Maybe I got a bad batch.",
 "Perfect for exam week. The delivery to the library locker is genius.",
 "The app logged me out every day for a week. Fixed now but annoying.",
 "Too expensive. Dunkin is right there.",
 "Delivery was on time, beans were fresh, app worked. Boring review, happy customer.",
 "I asked to skip a month and got charged anyway. Still waiting on the refund.",
 "Great taste, decent price for the quality. Wish the bag was resealable.",
 "Support ghosted me for 5 days about a missing delivery.",
 "Medium roast is excellent. Dark roast tastes like an ashtray.",
 "Cancelled. Not because of the coffee — the billing surprises were the last straw.",
 "The QR code on the bag that shows the farm it came from is a lovely touch.",
 "App is fine, coffee is fine, price is fine. Nothing special.",
 "Locker delivery failed three times this month. I'm on the edge.",
 "Way better than the dining hall coffee. Worth every dollar.",
 "Got someone else's order. Twice. Do you have a system??",
 "The pause button in the app doesn't do anything. Pressed it four times.",
 "Referred three friends. The referral credit never showed up.",
 "Beans are roasted the same week they ship. You can taste it.",
 "Meh. Fine for the price, I guess.",
 "The chat support person was rude when I asked about my late bag.",
 "Switched from the light roast to espresso blend. Life-changing.",
 "$24 would be fine if delivery actually came when it says it will.",
 "App redesign is beautiful but now I can't find my order history.",
 "Been a member since September. Genuinely the highlight of my Monday.",
]
print(f"{len(REVIEWS)} reviews loaded. Reading them yourself: ~10 minutes. Let's see how long the model takes.")

In [ ]:
# ── One call → 30 structured rows ────────────────────────────────
class ReviewTag(BaseModel):
    review_id: int = Field(description="1-based index of the review in the list")
    sentiment: Literal["positive", "neutral", "negative"]
    theme: Literal["taste", "price", "delivery", "app_bugs", "billing", "customer_service", "other"]
    churn_risk: bool = Field(description="True if the customer hints at cancelling or leaving")
    summary: str = Field(description="Under 8 words")

numbered = "\n".join(f"{i+1}. {r}" for i, r in enumerate(REVIEWS))
tags, usage = structured(
    f"Tag every one of these {len(REVIEWS)} customer reviews. Return exactly one object per review, in order.\n\n{numbered}",
    schema=list[ReviewTag])

df = pd.DataFrame([t.model_dump() for t in tags])
df["review"] = [REVIEWS[i-1] for i in df["review_id"]]
print(f"✅ {len(df)} rows back in one call  ·  {usage.prompt_token_count} tokens in / {usage.candidates_token_count} out\n")
display(df[["review_id", "sentiment", "theme", "churn_risk", "summary"]])

In [ ]:
# ── The chart the product team actually needs ────────────────────
import matplotlib.pyplot as plt

INK, MUTED, GRID = "#1f2937", "#6b7280", "#e5e7eb"
HUE = "#4f6ad8"          # one hue: this is a magnitude, not a rainbow

neg = df[df.sentiment == "negative"]
counts = neg.theme.value_counts().sort_values()

fig, ax = plt.subplots(figsize=(8, 4.2), dpi=110)
bars = ax.barh(counts.index.str.replace("_", " "), counts.values, color=HUE, height=0.55)
for b, v in zip(bars, counts.values):
    ax.text(b.get_width() + 0.15, b.get_y() + b.get_height()/2, str(v), va="center", color=INK, fontsize=11)
ax.set_title(f"What are unhappy BrewPass customers unhappy about?  (n={len(neg)} negative of {len(df)})",
             loc="left", color=INK, fontsize=12, pad=14)
ax.set_xlabel("negative reviews", color=MUTED)
ax.tick_params(colors=INK, length=0); ax.set_xticks([])
for s in ("top", "right", "bottom"): ax.spines[s].set_visible(False)
ax.spines["left"].set_color(GRID)
plt.tight_layout(); plt.show()

risk = df[df.churn_risk]
print(f"🚨 {len(risk)} customers at churn risk — the ones to call TODAY:")
for _, r in risk.iterrows():
    print(f"   #{r.review_id:>2} [{r.theme}] {r.summary}")

In [ ]:
# ── The CFO version: what did that cost, and what would 10,000 cost? ─
IN_PRICE, OUT_PRICE = 0.30, 2.50      # $ per 1M tokens, Gemini 2.5 Flash paid tier (free tier today: $0)
cost_30 = usage.prompt_token_count/1e6*IN_PRICE + usage.candidates_token_count/1e6*OUT_PRICE
per_review = cost_30 / len(REVIEWS)
print(f"30 reviews tagged for ${cost_30:.4f}  →  ${per_review*1000:.2f} per 1,000 reviews  →  ${per_review*10_000:.2f} per 10,000")
print("A junior analyst tagging 10,000 reviews by hand: roughly a week, and they'd tag #17 differently on Friday than on Monday.")

> 💼 **Boardroom line:** *"We tag every review, every ticket, every call transcript — consistently, for pocket change — and the chart updates itself."* You just built the pipeline behind that sentence. Loop it over a CSV and it's a product.

---

## 📸 Part 3 — Sight: paper → data

Gemini is natively **multimodal**: it reads images the same way it reads text. Take your phone out.

**Photograph one of these and upload it below:** a receipt · a business card · a whiteboard or notebook page with to-dos · a restaurant menu. No photo handy? Skip the upload dialog (click *Cancel*) and the cell generates a sample receipt for you.

Then pick the matching schema and watch the photo become a table.

In [ ]:
# ── Upload a photo (or Cancel to use the generated sample) ───────
from google.colab import files
from IPython.display import Image as IPImage, display
import io, mimetypes

DOC_TYPE = "receipt"          # ← change to "business_card" or "whiteboard" to match YOUR photo

uploaded = files.upload()
if uploaded:
    fname, img_bytes = next(iter(uploaded.items()))
    mime = mimetypes.guess_type(fname)[0] or "image/jpeg"
    print(f"📎 using your photo: {fname} ({mime}, {len(img_bytes)//1024} KB)")
else:
    # No photo? Draw a plausible receipt so the demo still works.
    from PIL import Image, ImageDraw, ImageFont
    W, H = 520, 640
    im = Image.new("RGB", (W, H), "#fbfaf6"); d = ImageDraw.Draw(im)
    try:
        f_b = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSansMono-Bold.ttf", 22)
        f_r = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSansMono.ttf", 20)
    except OSError:
        f_b = f_r = ImageFont.load_default()
    lines = [("TREMONT ST. CAFE", f_b), ("73 Tremont St, Boston MA", f_r), ("09/28/2026  12:04 PM   #A-3391", f_r), ("", f_r),
             ("2  Oat Latte (L)           11.50", f_r), ("1  Almond Croissant         4.25", f_r),
             ("3  Drip Coffee (M)          9.75", f_r), ("1  Breakfast Sandwich       8.95", f_r), ("", f_r),
             ("SUBTOTAL                   34.45", f_r), ("TAX 6.25%                   2.15", f_r),
             ("TOTAL                      36.60", f_b), ("", f_r), ("VISA ****4471   APPROVED", f_r), ("Thank you! Come again", f_r)]
    y = 40
    for txt, fnt in lines:
        d.text((40, y), txt, fill="#2b2b2b", font=fnt); y += 38
    buf = io.BytesIO(); im.save(buf, format="PNG")
    img_bytes, mime, fname = buf.getvalue(), "image/png", "generated_receipt.png"
    print("🧾 no upload — using a generated sample receipt")

display(IPImage(data=img_bytes, width=360))

In [ ]:
# ── The photo becomes a table ────────────────────────────────────
class LineItem(BaseModel):
    description: str
    quantity: int
    line_total: float

class Receipt(BaseModel):
    merchant: str
    date: str | None
    items: list[LineItem]
    subtotal: float | None
    tax: float | None
    total: float
    expense_category: Literal["meals", "travel", "office_supplies", "software", "other"]
    reimbursable_note: str = Field(description="One sentence a finance team would want")

class BusinessCard(BaseModel):
    full_name: str
    title: str | None
    company: str | None
    email: str | None
    phone: str | None
    linkedin_or_website: str | None
    suggested_followup: str = Field(description="One line: how to follow up with this person")

class Whiteboard(BaseModel):
    topic: str = Field(description="What this page/board is about, under 10 words")
    action_items: list[str]
    owners_mentioned: list[str]
    open_questions: list[str]
    deadline_mentions: list[str]

SCHEMAS = {"receipt": Receipt, "business_card": BusinessCard, "whiteboard": Whiteboard}
schema = SCHEMAS[DOC_TYPE]

result, usage = structured(
    [types.Part.from_bytes(data=img_bytes, mime_type=mime),
     f"Extract this {DOC_TYPE.replace('_',' ')} into the schema. Use null for anything not visible. Do not invent values."],
    schema=schema)

print(json.dumps(result.model_dump(), indent=2, ensure_ascii=False))

# Trust, but verify — a guardrail your finance team will insist on
if DOC_TYPE == "receipt" and result.items:
    items_sum = round(sum(i.line_total for i in result.items), 2)
    print(f"\n🧮 line items add to {items_sum} · subtotal read as {result.subtotal} · total read as {result.total}")
    print("   ✅ arithmetic checks out" if result.subtotal and abs(items_sum - result.subtotal) < 0.05
          else "   ⚠️ mismatch — a human should glance at this one (that flag IS the product)")
print(f"\ntokens: {usage.prompt_token_count} in (the photo is ~260 of them) / {usage.candidates_token_count} out")

> 💼 **Boardroom line:** *"Every receipt, invoice, and whiteboard photo in the company is already data — we just weren't reading it."*

---

## ✅ What you shipped today

| You built | The business word for it |
|-----------|--------------------------|
| A schema the model must fill in | **Structured output** — the bridge from AI to your systems |
| 30 reviews → a DataFrame → a chart, one call | **Batch enrichment** — the Monday-morning pipeline |
| A photo → JSON with a sanity check | **Document intelligence** — and the guardrail that makes finance trust it |

Three calls. Three things the chat window can't do. Same key you got Wednesday.

### 🔭 Where this goes
- **Wednesday (Session 4):** today *you* decided every step. Wednesday the **model** decides — which function to call, when, and whether it's done. Your first agent.
- **Your final project:** almost every strong project is one of today's three patterns pointed at a real dataset or a real stack of documents. Start noticing the paper and the prose around you.

### 🧪 Try before Wednesday (optional)
1. Change the `Ticket` schema: add a `language: str` field and paste an email in Spanish or Turkish.
2. Replace `REVIEWS` with 30 real App Store reviews of any app you use. Does the chart surprise you?
3. Photograph a menu and write a `Menu` schema: items, prices, and `vegetarian: bool`. Ask for the cheapest vegetarian option in one line of pandas.